In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

c:\Users\ALI\miniconda3\Lib\site-packages\torch\_subclasses\functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [7]:
pairs = []
counts = {}
def make_training_pairs(text):
    # 回傳一個 list，裡面是 (input字元, output字元) 的 tuple
    for i in range(len(text)-1):
        pairs.append((text[i],text[i+1]))
        if text[i] not in counts:
            counts[text[i]] = {}
        if text[i+1] not in counts[text[i]]:
            counts[text[i]][text[i+1]] = 0
        
        counts[text[i]][text[i+1]] += 1
       
make_training_pairs("the cat sat on the mat the cat ate the rat")
print(counts)

{'t': {'h': 4, ' ': 4, 'e': 1}, 'h': {'e': 4}, 'e': {' ': 5}, ' ': {'c': 2, 's': 1, 'o': 1, 't': 3, 'm': 1, 'a': 1, 'r': 1}, 'c': {'a': 2}, 'a': {'t': 6}, 's': {'a': 1}, 'o': {'n': 1}, 'n': {' ': 1}, 'm': {'a': 1}, 'r': {'a': 1}}


In [8]:
def generate(start_char, n):
    # 從 start_char 開始，生成 n 個字元
    for i in range(n-1):    
        print(max(counts[start_char], key=lambda k: counts[start_char][k]))
        start_char=max(counts[start_char], key=lambda k: counts[start_char][k])
generate("h",20)

e
 
t
h
e
 
t
h
e
 
t
h
e
 
t
h
e
 
t


In [9]:
def one_hot(char):
    vec = [0] * 26      # 建一個長度26全是0的list
    idx = ord(char) - ord('a')
    vec[idx] = 1        # 把對應位置設成1
    return vec

one_hot("b")

[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [10]:
X = []  # inputs
Y = []  # outputs

for input_char, output_char in pairs:
    print(output_char)
    if input_char.isalpha() and output_char.isalpha():
        X.append(one_hot(input_char))
        Y.append(ord(output_char) - ord('a'))  # output只需要ID，不需要one-hot

h
e
 
c
a
t
 
s
a
t
 
o
n
 
t
h
e
 
m
a
t
 
t
h
e
 
c
a
t
 
a
t
e
 
t
h
e
 
r
a
t


In [11]:
print(len(X),
X[0],
Y[0])

21 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0] 7


In [12]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [13]:
import torch
import torch.nn as nn
print(torch.__version__)

2.12.0+cpu


In [14]:
X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.long)

print(X_tensor.shape)
print(Y_tensor.shape)

torch.Size([21, 26])
torch.Size([21])


In [15]:
model = nn.Linear(26, 26)
output = model(X_tensor)
print(output.shape)

torch.Size([21, 26])


In [16]:
# 1. loss function：衡量預測有多錯
loss_fn = nn.CrossEntropyLoss()

# 2. optimizer：負責更新參數
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# 3. 訓練迴圈
for epoch in range(100):
    output = model(X_tensor)
    loss = loss_fn(output, Y_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"epoch {epoch}, loss: {loss.item():.4f}")

epoch 0, loss: 3.2083
epoch 10, loss: 2.8611
epoch 20, loss: 2.5433
epoch 30, loss: 2.2587
epoch 40, loss: 2.0092
epoch 50, loss: 1.7942
epoch 60, loss: 1.6110
epoch 70, loss: 1.4558
epoch 80, loss: 1.3243
epoch 90, loss: 1.2125


In [17]:
def generate_nn(start_char, n):
    current = start_char
    result = start_char
    
    for i in range(n):
        # 把current轉成one-hot tensor
        x = torch.tensor(one_hot(current), dtype=torch.float32)
        
        # 讓模型預測
        output = model(x)
        
        # 找機率最高的字元
        next_idx = output.argmax().item()
        next_char = chr(next_idx + ord('a'))
        
        result += next_char
        current = next_char
    
    return result

print(generate_nn('t', 20))

theatheatheatheatheat


In [18]:
generate('t',20)

h
e
 
t
h
e
 
t
h
e
 
t
h
e
 
t
h
e
 


In [19]:
import torch.nn.functional as F

# 假設每個字元用4維向量表示（簡化版）
torch.manual_seed(42)
Q = torch.randn(5, 4)  # 5個字元，每個4維
K = torch.randn(5, 4)
V = torch.randn(5, 4)

# 計算attention權重
scores = Q @ K.T        # 點積
weights = F.softmax(scores, dim=-1)  # 變成機率
output = weights @ V    # 加權平均

print(weights)

tensor([[6.2868e-02, 1.7510e-03, 2.2287e-01, 3.2972e-04, 7.1218e-01],
        [1.5593e-02, 3.5201e-01, 1.1839e-01, 5.0110e-01, 1.2906e-02],
        [2.6932e-01, 1.4982e-01, 1.1536e-01, 1.9541e-01, 2.7010e-01],
        [8.6699e-02, 1.6982e-01, 8.8891e-02, 5.6471e-01, 8.9878e-02],
        [1.6118e-01, 5.2632e-01, 2.4299e-01, 2.4076e-02, 4.5426e-02]])


In [20]:
embedding = nn.Embedding(26, 8)
idx = torch.tensor([19])  # 't' 的 index
vec = embedding(idx)
print(vec.shape)

torch.Size([1, 8])


In [21]:
text = "thecat"
indices = torch.tensor([ord(c) - ord('a') for c in text])

vecs = embedding(indices)
print(vecs.shape)

torch.Size([6, 8])
